# 5.5 — Code Optimization in a Training Loop

**Chapter 5, sections 5.9.3 and 5.10.3**, and the starting point for **Exercise 6**.

Rebuilt from `Notebooks old/Spark-Example-20a-Code-Optimization.ipynb`.

**The question this notebook answers:** the chapter's claim is that a distributed program can be
made several times faster without changing what it computes, by three edits that look like
nothing — a line moved out of a function, a column appended once instead of every iteration, and
an exponential written a different way. All three are in the inherited notebook. **None of them
is measured there**, and in a chapter about performance that is the omission that matters.

So this notebook leads with the measurement and then takes the three optimizations apart one at a
time, each with the quantity that shows what it did: jobs launched, calls to `np.append`,
evaluations of `np.exp`.

**What was left behind in the rebuild.** The source notebook spends thirty-one of its
thirty-five cells training six gradient-descent optimizers — SGD, Momentum, Nesterov, Adagrad,
RMSprop, Adam — and plotting their convergence. That is chapter 8's subject, not chapter 5's, and
carrying it here buried the four cells the chapter actually points at. Only SGD is used below.
The optimizer survey remains in the read-only library in
`Spark-Example-20a-Code-Optimization.ipynb` and `Spark-Example-20-Adam-Sgdm-with-Tree-Aggregate.ipynb`.

Runs on a laptop in about a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, json, time, tempfile, logging, urllib.request
from urllib.parse import urlparse
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-5.5")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 200)

_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

def jobs_so_far():
    return len(ui("/jobs"))

print("Spark", spark.version, "| numpy", np.__version__)

Spark 4.2.0 | numpy 2.5.2


## 1. The data and the two cached datasets

Unchanged from the source notebook: one hundred thousand points from two Gaussian blobs, a fixed
`random_state`, split nine to one. The two `cache()` calls are the ones the chapter reproduces at
§5.9.3, and the second of them is the interesting one — it is optimization 2, arriving before the
optimizations are discussed.

In [2]:
n_feature, n_components, n = 2, 2, 100_000
max_iter = 5

X, y = make_blobs(n_samples=n, centers=n_components, n_features=n_feature,
                  cluster_std=[0.5] * n_components, random_state=2)

rdd_X = sc.parallelize(X)
rdd_y = sc.parallelize(y)

# Split once, and hold the training data across all iterations.
traindata, testdata = rdd_y.zip(rdd_X).randomSplit([0.9, 0.1], seed=12345)
traindata.cache()

# The intercept is appended once, into a second cached dataset, rather than being
# recomputed for every record on every iteration.  <- optimization 2, in place
traindata1 = traindata.map(lambda x: (x[0], np.append(x[1], 1)))
traindata1.cache()

train_size = traindata.count()          # computed ONCE, outside -- optimization 1, in place
test_size = testdata.count()

print("a training record, without the intercept:", traindata.take(1))
print("a training record, with it appended     :", traindata1.take(1))
print(f"\ntraining records {train_size:,}   test records {test_size:,}")
print("both datasets are cached; the Storage tab entry is what makes the loop below cheap")

a training record, without the intercept: [(np.int64(0), array([-1.18378037, -9.66095522]))]
a training record, with it appended     : [(np.int64(0), array([-1.18378037, -9.66095522,  1.        ]))]

training records 90,122   test records 9,878
both datasets are cached; the Storage tab entry is what makes the loop below cheap


## 2. The measurement, before anything is taken apart

Both versions of the function are defined below exactly as the source notebook has them, and then
run against each other on the same data with the same seed. Two quantities are recorded that the
source notebook never records: **how many Spark jobs each call launches**, and **how long it
takes**.

The printing of the per-iteration cost is suppressed here so the two runs are comparable; nothing
else about either function is altered.

In [3]:
def LogisticRegression(traindata=None, max_iteration=max_iter, learningRate=0.01,
                       regularization=0.01, mini_batch_size=512, tolerance=10e-8,
                       verbose=False):
    """The original. Note line 4: a count() at the top of the function."""
    prev_cost, L_cost = 0, []
    train_size = traindata.count()                       # <-- an action, once per call
    parameter_size = len(traindata.take(1)[0][1]) + 1    # <-- another action, once per call
    np.random.seed(0)
    parameter_vector = np.random.normal(0, 0.1, parameter_size)

    for i in range(max_iteration):
        bc_weights = parameter_vector[:-1]
        bc1_weights = parameter_vector[-1]
        min_batch = traindata.sample(False, mini_batch_size / train_size, 1 + i)
        res = min_batch.treeAggregate(
            (np.zeros(parameter_size), 0, 0),
            lambda x, y: (x[0]
                          + (np.append(y[1], 1)) * (-y[0] + (np.exp(np.dot(y[1], bc_weights) + bc1_weights)
                          / (1 + np.exp(np.dot(y[1], bc_weights) + bc1_weights)))),
                          x[1]
                          + y[0] * (-(np.dot(y[1], bc_weights) + bc1_weights))
                          + np.log(1 + np.exp(np.dot(y[1], bc_weights) + bc1_weights)),
                          x[2] + 1),
            lambda x, y: (x[0] + y[0], x[1] + y[1], x[2] + y[2]))
        cost = res[1] / res[2] + regularization * (np.square(parameter_vector).sum())
        gradient = (1.0 / res[2]) * res[0] + 2 * regularization * parameter_vector
        parameter_vector = parameter_vector - learningRate * gradient
        if verbose:
            print("Iteration No.", i, " Cost=", cost)
        if abs(cost - prev_cost) < tolerance:
            break
        prev_cost = cost
        L_cost.append(cost)
    return parameter_vector, L_cost


def LogisticRegression_optimized(traindata=None, max_iteration=max_iter, learningRate=0.01,
                                 regularization=0.01, mini_batch_size=512, tolerance=10e-8,
                                 train_size=1, verbose=False):
    """The same computation, with the three optimizations applied."""
    prev_cost, L_cost = 0, []
    # Optimization 1: no count() here. train_size arrives as a parameter.
    # Optimization 2: the intercept is already in the vector, so no "+ 1".
    parameter_size = len(traindata.take(1)[0][1])
    np.random.seed(0)
    parameter_vector = np.random.normal(0, 0.1, parameter_size)

    for i in range(max_iteration):
        bc_weights = parameter_vector
        min_batch = traindata.sample(False, mini_batch_size / train_size, 1 + i)
        # Optimization 3: the sigmoid as 1/(1+exp(-z)) rather than exp(z)/(1+exp(z)).
        res = min_batch.treeAggregate(
            (np.zeros(parameter_size), 0, 0),
            lambda x, y: (x[0]
                          + (y[1]) * (-y[0] + (1 / (np.exp(-np.dot(y[1], bc_weights)) + 1))),
                          x[1]
                          + y[0] * (-(np.dot(y[1], bc_weights)))
                          + np.log(1 + np.exp(np.dot(y[1], bc_weights))),
                          x[2] + 1),
            lambda x, y: (x[0] + y[0], x[1] + y[1], x[2] + y[2]))
        cost = res[1] / res[2] + regularization * (np.square(parameter_vector).sum())
        gradient = (1.0 / res[2]) * res[0] + 2 * regularization * parameter_vector
        parameter_vector = parameter_vector - learningRate * gradient
        if verbose:
            print("Iteration No.", i, " Cost=", cost)
        if abs(cost - prev_cost) < tolerance:
            break
        prev_cost = cost
        L_cost.append(cost)
    return parameter_vector, L_cost

print("both functions defined")

both functions defined


In [4]:
BATCH = 50_000

def run(label, fn, **kw):
    before = jobs_so_far()
    sc.setJobDescription(label)
    t0 = time.time()
    theta, costs = fn(max_iteration=max_iter, learningRate=0.01,
                      regularization=0, mini_batch_size=BATCH, **kw)
    secs = time.time() - t0
    return {"version": label, "jobs": jobs_so_far() - before,
            "seconds": round(secs, 2), "final cost": round(costs[-1], 6)}, theta

orig_row, theta_orig = run("original", LogisticRegression, traindata=traindata)
opt_row,  theta_opt  = run("optimized", LogisticRegression_optimized,
                           traindata=traindata1, train_size=train_size)

print(pd.DataFrame([orig_row, opt_row]).to_string(index=False))
print(f"\noriginal  theta = {np.round(theta_orig, 6)}")
print(f"optimized theta = {np.round(theta_opt, 6)}")
print(f"largest difference between them: {np.abs(theta_orig - theta_opt).max():.2e}")
assert np.allclose(theta_orig, theta_opt, atol=1e-8), "the two versions no longer agree"
print("\nThe two versions compute the same model. Everything below is about what they cost.")

  version  jobs  seconds  final cost
 original     7     0.62    0.461797
optimized     6     0.41    0.461797

original  theta = [0.197308 0.10025  0.10127 ]
optimized theta = [0.197308 0.10025  0.10127 ]
largest difference between them: 0.00e+00

The two versions compute the same model. Everything below is about what they cost.


The assertion is the point of running both. An optimization that changes the answer is not an
optimization, and a sentence claiming the two forms are equivalent is worth less than a line that
fails if they ever stop being.

## 3. Optimization 1: the action that looked like initialization

`train_size = traindata.count()` sits at the top of the function, in the position a reader scans
past. It is an action, so it launches a full distributed job over the training set — **once per
call**, not once per program.

The extra job is visible in the table above. What the chapter asks for in Exercise 6(b) is the
total across a comparison of several optimizers, so the per-call cost is what has to be measured.

In [5]:
jobs_per_call_orig = orig_row["jobs"]
jobs_per_call_opt = opt_row["jobs"]
extra = jobs_per_call_orig - jobs_per_call_opt

print(f"jobs per call, original  : {jobs_per_call_orig}")
print(f"jobs per call, optimized : {jobs_per_call_opt}")
print(f"difference               : {extra}")
print()
print("Where they come from, per call:")
print(f"   {max_iter:>2} treeAggregate, one per iteration  -- both versions")
print( "    1 take(1), to size the parameter vector  -- both versions")
print( "    1 count(), to size the training set      -- ORIGINAL ONLY")
print()

# Exercise 6(b): the source notebook compares six optimizers, calling the function once each.
OPTIMIZERS = 6
print(f"Exercise 6(b): the source notebook calls this function once for each of"
      f" {OPTIMIZERS} optimizers.")
print(f"   extra jobs from the un-hoisted count(): {OPTIMIZERS} x {extra} = {OPTIMIZERS*extra}")
print(f"   each one a full pass over {train_size:,} training records that computes a number")
print(f"   already known before the function was entered.")
print()
print("In the Jobs tab this is the reading the chapter puts first: more jobs than the program")
print("appears to have actions. Nothing in the source of the function looks like an action.")

jobs per call, original  : 7
jobs per call, optimized : 6
difference               : 1

Where they come from, per call:
    5 treeAggregate, one per iteration  -- both versions
    1 take(1), to size the parameter vector  -- both versions
    1 count(), to size the training set      -- ORIGINAL ONLY

Exercise 6(b): the source notebook calls this function once for each of 6 optimizers.
   extra jobs from the un-hoisted count(): 6 x 1 = 6
   each one a full pass over 90,122 training records that computes a number
   already known before the function was entered.

In the Jobs tab this is the reading the chapter puts first: more jobs than the program
appears to have actions. Nothing in the source of the function looks like an action.


## 4. Optimization 2: the intercept, appended once instead of every time

The original appends the intercept inside the aggregation: `np.append(y[1], 1)` runs in the
`seqOp`, which is executed **once per record per iteration**. The optimized version appends it
once, into `traindata1`, which is cached — so the work happens a single time for the whole run and
is then reused by every iteration.

This is the second `cache()` of §5.9.3, and the chapter singles it out as the more instructive of
the two: appending a column is cheap, so caching the result looks wasteful until the multiplier is
written down.

In [6]:
records_per_batch = BATCH
appends_original = records_per_batch * max_iter
appends_optimized = train_size          # once per record, when traindata1 was built

print(f"records touched per iteration (the mini-batch) : {records_per_batch:,}")
print(f"iterations                                     : {max_iter}")
print()
print(f"np.append calls, original  : {records_per_batch:,} x {max_iter}"
      f" = {appends_original:,}   (inside the seqOp, every iteration)")
print(f"np.append calls, optimized : {appends_optimized:,}"
      f"   (once, when traindata1 was built and cached)")
print()
print(f"With the {OPTIMIZERS} optimizer comparisons of the source notebook, the original repeats")
print(f"the append {OPTIMIZERS * appends_original:,} times against"
      f" {appends_optimized:,} for the optimized version.")
print()

# What one append actually costs, so the multiplier is attached to a real number.
row = traindata.take(1)[0][1]
reps = 200_000
t0 = time.time()
for _ in range(reps):
    np.append(row, 1)
per_append_us = (time.time() - t0) / reps * 1e6
print(f"measured cost of one np.append on this vector : {per_append_us:.2f} microseconds")
print(f"   original,  one optimizer run : {appends_original * per_append_us / 1e6:6.2f} s of append")
print(f"   optimized, one optimizer run : {appends_optimized * per_append_us / 1e6:6.2f} s of append,"
      f" paid once and cached")

records touched per iteration (the mini-batch) : 50,000
iterations                                     : 5

np.append calls, original  : 50,000 x 5 = 250,000   (inside the seqOp, every iteration)
np.append calls, optimized : 90,122   (once, when traindata1 was built and cached)

With the 6 optimizer comparisons of the source notebook, the original repeats
the append 1,500,000 times against 90,122 for the optimized version.

measured cost of one np.append on this vector : 0.54 microseconds
   original,  one optimizer run :   0.14 s of append
   optimized, one optimizer run :   0.05 s of append, paid once and cached


Note what the second line is really saying. The optimized version does *more* appends in absolute
terms on a single run, because it appends to every training record rather than only to the records
that happen to be sampled. It wins because it does them **once** rather than once per iteration
per optimizer, and because the result is cached rather than recomputed. The general rule the
chapter gives is exactly this: *the value of a cache is the cost of the work it prevents,
multiplied by the number of times that work would otherwise be repeated.*

## 5. Optimization 3: counting the exponentials

§5.10.3 says the sigmoid was originally written so that *"the exponential was evaluated three
times per record, where an algebraically identical form evaluates it once."* That is a countable
claim, and the cell below counts it — by replacing `np.exp` with a wrapper that tallies its calls
and running each `seqOp` once on a single record.

In [7]:
class ExpCounter:
    """np.exp, with a tally."""
    def __init__(self):
        self.n = 0
    def __call__(self, z):
        self.n += 1
        return np.exp(z)

def original_seq_op(exp, acc, rec, w, b):
    """The original seqOp, verbatim, with np.exp routed through the counter."""
    return (acc[0] + (np.append(rec[1], 1)) * (-rec[0] + (exp(np.dot(rec[1], w) + b)
                                                          / (1 + exp(np.dot(rec[1], w) + b)))),
            acc[1] + rec[0] * (-(np.dot(rec[1], w) + b))
                   + np.log(1 + exp(np.dot(rec[1], w) + b)),
            acc[2] + 1)

def optimized_seq_op(exp, acc, rec, w):
    """The optimized seqOp, verbatim, with np.exp routed through the counter."""
    return (acc[0] + (rec[1]) * (-rec[0] + (1 / (exp(-np.dot(rec[1], w)) + 1))),
            acc[1] + rec[0] * (-(np.dot(rec[1], w)))
                   + np.log(1 + exp(np.dot(rec[1], w))),
            acc[2] + 1)

rec_plain = traindata.take(1)[0]
rec_with_intercept = traindata1.take(1)[0]
w3, w2, b = theta_opt, theta_opt[:-1], theta_opt[-1]

c1 = ExpCounter(); original_seq_op(c1, (np.zeros(3), 0, 0), rec_plain, w2, b)
c2 = ExpCounter(); optimized_seq_op(c2, (np.zeros(3), 0, 0), rec_with_intercept, w3)

print(pd.DataFrame([
    {"seqOp": "original", "np.exp calls per record": c1.n},
    {"seqOp": "optimized", "np.exp calls per record": c2.n},
]).to_string(index=False))

print(f"\nOf those, the ones belonging to the sigmoid itself:")
print(f"   original  exp(z) / (1 + exp(z))  -> 2")
print(f"   optimized 1 / (1 + exp(-z))      -> 1")
print(f"   the remaining call in each is the log-loss term, log(1 + exp(z)),")
print(f"   which the rewrite does not touch.")
print()
print(f"per record per iteration, the rewrite saves {c1.n - c2.n} evaluation of exp;")
print(f"over {BATCH:,} sampled records and {max_iter} iterations that is"
      f" {(c1.n - c2.n) * BATCH * max_iter:,} evaluations for one optimizer run.")

    seqOp  np.exp calls per record
 original                        3
optimized                        2

Of those, the ones belonging to the sigmoid itself:
   original  exp(z) / (1 + exp(z))  -> 2
   optimized 1 / (1 + exp(-z))      -> 1
   the remaining call in each is the log-loss term, log(1 + exp(z)),
   which the rewrite does not touch.

per record per iteration, the rewrite saves 1 evaluation of exp;
over 50,000 sampled records and 5 iterations that is 250,000 evaluations for one optimizer run.


**The chapter's figure needs one word changed.** The count is three against two for the whole
`seqOp`, and two against one for the sigmoid. Both halves of the chapter's sentence are true of
something, but not of the same thing: "three times" counts the exponentials in the entire
aggregation function, and "once" counts those in the sigmoid alone. The log-loss term evaluates an
exponential in both versions and is untouched by the rewrite.

The lesson survives the correction intact, and the chapter states it well: *the innermost function
of a distributed job is executed once per record per iteration, so ordinary arithmetic economies
there are multiplied by a very large number.* Neither of these is a Spark technique. What Spark
supplies is the multiplier.

## 6. What the three add up to

The three are applied together in the source notebook and are taken apart here only in the
counting above; the table below is the two end points, all three applied against none of them.

In [8]:
rows = []
for label, fn, kw in [
    ("all three applied", LogisticRegression_optimized,
     dict(traindata=traindata1, train_size=train_size)),
    ("none of the three (the original)", LogisticRegression,
     dict(traindata=traindata)),
]:
    row, _ = run(label, fn, **kw)
    row["version"] = label
    rows.append(row)

print(pd.DataFrame(rows)[["version", "jobs", "seconds", "final cost"]].to_string(index=False))

print("\nIn local mode on a hundred thousand points the clock barely separates them, and saying so")
print("is more useful than quoting a speedup this machine cannot support. What does separate them")
print("is countable and does not depend on the machine:")
print(f"   jobs launched per call         : {jobs_per_call_orig} -> {jobs_per_call_opt}")
print(f"   np.append per optimizer run    : {appends_original:,} -> {appends_optimized:,}, cached")
print(f"   np.exp per record per iteration: {c1.n} -> {c2.n}")
print("\nEach of the three is a constant factor on the innermost loop of a distributed job.")
print("On this laptop they are a rounding error; on a cluster and a real dataset they are the")
print("difference the chapter describes, and they cost three edits.")

                         version  jobs  seconds  final cost
               all three applied     6     0.36    0.461797
none of the three (the original)     7     0.38    0.461797

In local mode on a hundred thousand points the clock barely separates them, and saying so
is more useful than quoting a speedup this machine cannot support. What does separate them
is countable and does not depend on the machine:
   jobs launched per call         : 7 -> 6
   np.append per optimizer run    : 250,000 -> 90,122, cached
   np.exp per record per iteration: 3 -> 2

Each of the three is a constant factor on the innermost loop of a distributed job.
On this laptop they are a rounding error; on a cluster and a real dataset they are the
difference the chapter describes, and they cost three edits.


## Conclusion

The three edits are ordinary programming, and that is the chapter's point: almost every slow Spark
program is slow for one of three reasons, and this notebook is an instance of the third — it
repeated work that had already been done.

What this notebook establishes by running it:

1. **The two versions compute the same model**, asserted rather than claimed, so any later edit
   that breaks the equivalence fails loudly.
2. **A `count()` at the top of a function is a distributed job per call.** It reads as
   initialization and costs a full pass over the training data; across the source notebook's six
   optimizer comparisons that is six full passes computing a number that was already known.
3. **The intercept append moved from once per record per iteration to once per record, cached.**
   The optimized version performs more appends on a single pass and far fewer over a run, which is
   the caching rule in miniature.
4. **The sigmoid rewrite saves one exponential per record per iteration**, counted directly. The
   chapter's "three to one" is "three to two" for the whole aggregation and "two to one" for the
   sigmoid; the correction does not touch the argument.
5. **The wall clock is the wrong instrument at this scale**, and the countable quantities are the
   right one. Jobs, appends and exponentials are properties of the program; seconds are a property
   of the laptop.

*Chapter sections:* §5.9.3 (the two cache calls), §5.10.3 (repeating work already performed).
*Exercise 6* is answered in sections 3, 4 and 5: (a) by the job counts, (b) by the arithmetic over
six optimizers, and (c) by either of the other two optimizations, both of which belong to the same
category — repeating work already performed.